# 🚰 Notebook 2: Leaky Bucket

**Leaky bucket** = a queue that **drains at a fixed rate**. Requests pile up
(up to a cap) but are *served* at a steady tempo. Useful when a downstream
needs **smooth** traffic (no bursts at all).

### Mental picture 🎨
```
  incoming requests (bursty)
        │ │ │ │ │
        ▼ ▼ ▼ ▼ ▼
     ┌─────────────┐
     │ queue       │  capacity B (drop when full)
     └──────┬──────┘
            │  drains at leak_rate, no matter what
            ▼
        [ API ]
```


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## ❌ Bad first try — unbounded queue

If you forget the capacity cap, a burst will buffer *forever*, using memory
and adding latency. Real servers OOM this way. Always bound the queue.


In [ ]:
import time
from collections import deque

class UnboundedLeakyBucket:
    def __init__(self, leak_rate):
        self.leak_rate = leak_rate
        self.q = deque()

    def submit(self, req):
        self.q.append(req)     # no cap → unbounded memory!

    def drain_one(self):
        if self.q:
            return self.q.popleft()
        return None

bad = UnboundedLeakyBucket(leak_rate=5)
for i in range(100_000):
    bad.submit(i)
print(f"queued: {len(bad.q)} requests — in production this would eat RAM")

## ✅ Better: bounded queue + time-based drain

We don't actually run a thread; we compute how many requests *should have*
drained since the last call and shrink the queue accordingly. This is the
same "lazy" trick we used for token bucket.


In [ ]:
class LeakyBucket:
    def __init__(self, leak_rate, capacity):
        self.leak_rate = leak_rate   # requests served per second
        self.capacity = capacity     # max queued at once
        self.queued = 0.0            # using a float so partial leaks work
        self.last = time.monotonic()

    def allow(self):
        now = time.monotonic()
        # Leak whatever should have drained since last check.
        self.queued = max(0.0, self.queued - (now - self.last) * self.leak_rate)
        self.last = now
        if self.queued + 1 <= self.capacity:
            self.queued += 1
            return True
        return False

lb = LeakyBucket(leak_rate=5, capacity=10)
burst = [int(lb.allow()) for _ in range(20)]
print('burst result:', burst, '— allowed', sum(burst))

## 📊 Token vs Leaky: see the difference

Same input (20 requests in a tight burst, then a gap, then another burst).
- Token bucket **spends all its tokens** on the first burst (fast output).
- Leaky bucket **paces** the output across time.


In [ ]:
import matplotlib.pyplot as plt

class TokenBucket:  # re-declare so this cell is standalone
    def __init__(self, rate, capacity):
        self.rate = rate; self.capacity = capacity
        self.tokens = capacity; self.last = time.monotonic()
    def allow(self):
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1; return True
        return False

def simulate(limiter, arrival_times):
    allowed_times = []
    t0 = time.monotonic()
    for at in arrival_times:
        # wait until the scheduled arrival
        while time.monotonic() - t0 < at:
            time.sleep(0.001)
        if limiter.allow():
            allowed_times.append(at)
    return allowed_times

# arrivals: 20 in first 0.1s, silence, 20 more at t=1.5s
arrivals = [i * 0.005 for i in range(20)] + [1.5 + i * 0.005 for i in range(20)]

tb_out = simulate(TokenBucket(rate=5, capacity=10), arrivals)
lb_out = simulate(LeakyBucket(leak_rate=5, capacity=10), arrivals)

fig, axes = plt.subplots(2, 1, figsize=(9, 3.5), sharex=True)
axes[0].eventplot(arrivals, colors='gray'); axes[0].set_title('arrivals (bursty)')
axes[1].eventplot(tb_out, colors='green', lineoffsets=1, linelengths=0.8)
axes[1].eventplot(lb_out, colors='blue', lineoffsets=0, linelengths=0.8)
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['leaky out', 'token out'])
axes[1].set_xlabel('time (s)')
plt.tight_layout(); plt.show()
print(f"token allowed: {len(tb_out)}  |  leaky allowed: {len(lb_out)}")

## 🧠 Token vs Leaky — quick reference

| | Token bucket | Leaky bucket |
|---|---|---|
| Allows bursts? | yes (up to bucket size) | no — output is paced |
| Rejects when? | bucket empty | queue full |
| Output rate | can match arrival (up to burst) | constant `leak_rate` |
| Best for | user-facing APIs, quotas | shaping traffic to fragile downstreams |

### Real-world examples
- **NGINX `limit_req`** — leaky bucket (with optional `burst=` → hybrid).
- **Network routers** — ATM/Ethernet traffic shaping.
- **Background job queues** — throttle dispatching to a slow worker.
